<a href="https://colab.research.google.com/github/Godstouch/GNN-Student-Risk-Prediction-/blob/main/Graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install torch-geometric
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/Real_school_data_with_proxy_corrected.csv')
np.random.seed(42)
torch.manual_seed(42)

# Define leakage columns to EXCLUDE from features (X)

week_cols = [f'Week{i}_attendance' for i in range(1, 15)]

LEAKAGE_COLS = (
    # the 5 domain scores + intermediate cluster/label columns
    ['academic_domain', 'attendance_domain', 'socioeconomic_domain',
     'engagement_domain', 'accessibility_domain', 'cluster', 'dropout_risk']
    # raw components that feed academic_domain
    + ['Semester 1 average', 'Semester 2 average', 'Semester difference']
    # raw components that feed attendance_domain
    + week_cols + ['attendance_rate', 'low_attendance_weeks']
    # raw components that feed socioeconomic_domain
    + ['Household income level (standardized)', 'income_score',
       'Family dropout history', 'family_dropout_penalty',
       'Child labor involvement', 'child_labor_penalty']
    # raw components that feed engagement_domain
    + ['Teacher relationship quality', 'teacher_rel_score',
       'Peer relationship quality', 'peer_rel_score',
       'Extra-curricular activities', 'has_extracurricular']
    # raw components that feed accessibility_domain
    + ['long_commute_flag', 'long_walk_flag', 'Travel time to school (minutes)']
)



id_cols = ['Student ID']  # never used as a feature
label_col = 'dropout_risk'

feature_cols = [c for c in df.columns
                if c not in LEAKAGE_COLS + id_cols + [label_col]]

print(f"Excluded {len(LEAKAGE_COLS)} leakage columns.")
print(f"Remaining feature columns ({len(feature_cols)}):")
print(feature_cols)

# Encode categoricals, scale numerics

X_df = df[feature_cols].copy()

cat_cols = X_df.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_df.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()


X_df = pd.get_dummies(X_df, columns=cat_cols, dummy_na=True)

# Fill any remaining NaNs with 0
X_df = X_df.fillna(0)

# Scale all numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values.astype(float))

print(f"\nFinal feature matrix shape: {X_scaled.shape}")

# Encoding labels

label_order = ['Low', 'Medium', 'High']  # fix consistent class index mapping
le = LabelEncoder()
le.fit(label_order)
y = le.transform(df[label_col])
print(f"\nLabel classes (index order): {list(le.classes_)}")
print(f"Class counts: {np.bincount(y)}")

# Building edges — hybrid: school/class-group edges + KNN similarity

edges = set()

# Group edges: students sharing School + Class level are connected
group_key = df['School'].astype(str) + '_' + df['Class level'].astype(str)
for _, group_df in df.groupby(group_key):
    idxs = group_df.index.tolist()
    if len(idxs) > 1:
        # connect each student to a few classmates this step is to make the graph edges less dense
        for i in idxs:
            others = [j for j in idxs if j != i]
            sampled = np.random.choice(others, size=min(5, len(others)), replace=False)
            for j in sampled:
                edges.add((min(i, j), max(i, j)))

# Group edges: students sharing Section (extracurricular house), where present
section_df = df[df['Section'].notna()]
for _, group_df in section_df.groupby('Section'):
    idxs = group_df.index.tolist()
    if len(idxs) > 1:
        for i in idxs:
            others = [j for j in idxs if j != i]
            sampled = np.random.choice(others, size=min(3, len(others)), replace=False)
            for j in sampled:
                edges.add((min(i, j), max(i, j)))

# KNN similarity edges on the leakage-free feature space
k = 8
nbrs = NearestNeighbors(n_neighbors=k + 1).fit(X_scaled)
_, indices = nbrs.kneighbors(X_scaled)
for i, neighbors in enumerate(indices):
    for j in neighbors[1:]:  # skip self (index 0)
        edges.add((min(i, j), max(i, j)))

edge_list = list(edges)
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
# make undirected (add reverse direction)
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

print(f"\nTotal unique edges (directed pairs before symmetrizing): {len(edge_list)}")
print(f"Final edge_index shape: {edge_index.shape}")


x = torch.tensor(X_scaled, dtype=torch.float)
y_tensor = torch.tensor(y, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y_tensor)
print(f"\nGraph summary: {data}")

#70/15/15 split
n = data.num_nodes
indices = np.arange(n)

train_idx, temp_idx = train_test_split(
    indices, train_size=700, stratify=y, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, train_size=150, stratify=y[temp_idx], random_state=42
)

train_mask = torch.zeros(n, dtype=torch.bool)
val_mask = torch.zeros(n, dtype=torch.bool)
test_mask = torch.zeros(n, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"\nSplit sizes -> train: {train_mask.sum().item()}, "
      f"val: {val_mask.sum().item()}, test: {test_mask.sum().item()}")
print("Train class balance:", np.bincount(y[train_idx]))
print("Val class balance:  ", np.bincount(y[val_idx]))
print("Test class balance: ", np.bincount(y[test_idx]))


torch.save(data, 'real_graph_corrected.pt')


Excluded 41 leakage columns.
Remaining feature columns (12):
['School', 'Gender', 'Class level', 'Parental educational level', 'Household income level', 'Travel time to school', 'Mode of transport', 'Section', 'grade_number', 'stream_letter', 'subgroup', 'level_prefix']

Final feature matrix shape: (1000, 131)

Label classes (index order): [np.str_('High'), np.str_('Low'), np.str_('Medium')]
Class counts: [250 403 347]

Total unique edges (directed pairs before symmetrizing): 9216
Final edge_index shape: torch.Size([2, 18432])

Graph summary: Data(x=[1000, 131], edge_index=[2, 18432], y=[1000])

Split sizes -> train: 700, val: 150, test: 150
Train class balance: [175 282 243]
Val class balance:   [37 61 52]
Test class balance:  [38 60 52]

Saved: real_graph_corrected.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
